# V10: pilot-conditioned unfolded model (P=10)

V10 starts from each matching V9 checkpoint and adds one pilot-conditioning feature: standardized log condition number of the pilot matrix. Its six new per-layer modulation coefficients start at zero, so the initial V10 predictions must match V9. V10 keeps the 40-layer unfolded backbone and the V9 loss weights. It uses training and validation data only; **this notebook does not open or score a test split**.

Use the Google account whose Drive contains the V9 checkpoints, P=10 train/validation files, and cached GS validation curve. The script can run on CPU; a GPU only reduces runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import os, torch
root = '/content/gdrive/MyDrive'
required = [
    f'{root}/DL_dataset_RSR25dB/{split}_P10.mat'
    for split in ('train', 'validation')
] + [
    f'{root}/DL_dataset_RSR25dB/v6_starting_models/gs_validation_P10.json'
] + [
    f'{root}/unfolded_dl_v9_highsnr_finetune_p10_results/model_P10_seed{seed}_v9.pt'
    for seed in (20261026, 20261027, 20261028)
]
missing = [path for path in required if not os.path.isfile(path)]
assert not missing, f'Missing Drive files: {missing}'
print('Inputs found: P=10 train/validation data, validation GS curve, and V9 models.')
print('Runtime:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Upload the scripts
Run the next cell and upload all seven scripts from your workspace `v2` folder: `colab_unfolded_dl_v3_highsnr.py`, `colab_unfolded_dl_v5_snr_aware.py`, `colab_unfolded_dl_v6_gs_gap.py`, `colab_unfolded_dl_v7_continue_p10.py`, `colab_unfolded_dl_v8_continue_p10.py`, `colab_unfolded_dl_v9_highsnr_finetune.py`, and `colab_unfolded_dl_v10_pilot_conditioned_p10.py`.

In [ ]:
from google.colab import files
uploaded = files.upload()
required_scripts = {
    'colab_unfolded_dl_v3_highsnr.py',
    'colab_unfolded_dl_v5_snr_aware.py',
    'colab_unfolded_dl_v6_gs_gap.py',
    'colab_unfolded_dl_v7_continue_p10.py',
    'colab_unfolded_dl_v8_continue_p10.py',
    'colab_unfolded_dl_v9_highsnr_finetune.py',
    'colab_unfolded_dl_v10_pilot_conditioned_p10.py',
}
missing = required_scripts.difference(uploaded)
assert not missing, f'Upload these scripts: {sorted(missing)}'
print('All seven scripts uploaded.')

## Train
Run the cell below. It saves resumable checkpoints to Drive after each epoch and selects the checkpoint with the smallest worst validation DL-minus-GS gap across all eight SNR points. The initialization check prints the maximum difference between V9 and zero-initialized V10 predictions; it should be zero or extremely close to zero.

In [ ]:
!python /content/colab_unfolded_dl_v10_pilot_conditioned_p10.py

## Compare validation gaps
This compares V9 and V10 on validation only. Negative gaps favor DL. Do not use the old test split to select or tune V10.

In [ ]:
import json, numpy as np
root = '/content/gdrive/MyDrive'
v9_path = f'{root}/unfolded_dl_v9_highsnr_finetune_p10_results/summary_unfolded_v9_highsnr_p10.json'
v10_path = f'{root}/unfolded_dl_v10_pilot_conditioned_p10_results/summary_unfolded_v10_pilot_condition_p10.json'
with open(v9_path, encoding='utf-8') as f: v9 = json.load(f)
with open(v10_path, encoding='utf-8') as f: v10 = json.load(f)
snrs = [-5, 0, 5, 10, 15, 20, 25, 30]
print('SNR | V9 mean validation DL-GS | V10 mean validation DL-GS | V10 seeds beating GS')
for snr in snrs:
    key = str(snr)
    g9 = [r['selected_validation_dl_minus_gs_db_by_snr'][key] for r in v9]
    g10 = [r['selected_validation_dl_minus_gs_db_by_snr'][key] for r in v10]
    print(f'{snr:>3} dB | {np.mean(g9):+8.4f} dB | {np.mean(g10):+8.4f} dB | {sum(g < 0 for g in g10)}/3')
print('V10 validation results:', f'{root}/unfolded_dl_v10_pilot_conditioned_p10_results')

## After validation selection
If V10 improves the worst validation gap without a harmful tradeoff elsewhere, freeze the model and settings. Then create a new independent test split and compute its matching GS reference before making a final all-SNR claim. The existing test split has already informed earlier model changes.